# FMA-full shards

Browse the FMA-full audio on the music service's project storage (PD) and listen to single
tracks, read straight from the shards without unpacking them. The notebook writes nothing.

- `fma/fma_full/NNN.tar`: 156 verified tar shards, one per FMA folder, with the original MP3s.
- `manifests/fma_full_inventory.csv`: one row per track with its shard, path in the shard,
  size and SHA-1.
- `fma/fma_metadata/tracks.csv`: the published FMA metadata (title, artist, genre).

Run it in JupyterLab on Open OnDemand with the "Music Atlas" kernel, as described in
`docs/DEVELOPMENT.md`. Paths come from `ATLAS_DATA_DIR`, so a local copy of the data works too.

In [ ]:
import io
import tarfile
from functools import lru_cache

import pandas as pd
from IPython.display import Audio, display

from music_atlas.loader import load_windows
from music_atlas.settings import paths, profile

data = paths()["data"]


@lru_cache(maxsize=2)
def open_shard(shard: str) -> tarfile.TarFile:
    # Opening reads every tar header once (about 15 s on Lustre); later tracks are fast.
    return tarfile.open(data / shard)


inventory = pd.read_csv(data / "manifests/fma_full_inventory.csv")
per_shard = inventory.groupby("shard").agg(tracks=("track_id", "size"), gib=("size_bytes", "sum"))
per_shard["gib"] /= 2**30
print(f"{len(per_shard)} shards, {len(inventory)} tracks, {per_shard.gib.sum():.0f} GiB of MP3")
print(inventory.status.value_counts().to_dict())
per_shard.describe().round(1)

In [ ]:
# tracks.csv has a two-row header such as (track, title); loading it takes about 30 s.
tracks = pd.read_csv(
    data / "fma/fma_metadata/tracks.csv", index_col=0, header=[0, 1], low_memory=False
)
meta = pd.DataFrame(
    {
        "title": tracks["track", "title"],
        "artist": tracks["artist", "name"],
        "genre_top": tracks["track", "genre_top"],
        "duration_s": tracks["track", "duration"],
    }
)
catalogue = inventory.join(meta, on="track_id").set_index("track_id")
catalogue.genre_top.value_counts(dropna=False).head(10)

## Pick a shard

Shards are named after the FMA folders `000` to `155`; track `077047` lives in `077.tar`.
Set `SHARD` and run the cell again.

In [ ]:
SHARD = "077"

in_shard = catalogue[catalogue.shard == f"fma/fma_full/{SHARD}.tar"]
print(f"{len(in_shard)} tracks in {SHARD}.tar")
in_shard[["title", "artist", "genre_top", "duration_s"]].head(20)

## Listen to a track

Set `TRACK_ID` to an id from the table. The MP3 is read from its shard into memory and
decoded by the shared loader (`music_atlas.loader`), which cuts the windows the experiments
use: one centred window in the `quick` profile, beginning, middle and end in `full`.

In [ ]:
TRACK_ID = int(in_shard.index[0])
PROFILE = "quick"

row = catalogue.loc[TRACK_ID]
mp3 = open_shard(row.shard).extractfile(row.audio_path).read()
settings = profile(PROFILE)
windows = load_windows(
    io.BytesIO(mp3),
    TRACK_ID,
    sample_rate=settings["sample_rate"],
    window_seconds=settings["window_seconds"],
    windows_per_track=settings["windows_per_track"],
)
print(f"{TRACK_ID}: {row.title!r} by {row.artist} ({row.genre_top}), {row.duration_s} s")
for window in windows:
    print(f"window {window.index}: {window.start_seconds:.1f}-{window.end_seconds:.1f} s")
    display(Audio(window.samples, rate=window.sample_rate))